# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Wanoleo/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [36]:
%pip -q install duckdb huggingface_hub

In [37]:
import os
import getpass

HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass

if not HF_TOKEN:
    HF_TOKEN = getpass.getpass(
        "Paste your Hugging Face READ token (hf_...): "
    )

In [38]:
import duckdb

con = duckdb.connect()

con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

REL = "hf://datasets/FlyRank/internship-warehouse"

TABLES = {
    "dim_clients":
        f"read_parquet('{REL}/dim_clients.parquet')",

    "dim_content":
        f"read_parquet('{REL}/dim_content.parquet')",

    "fact_daily":
        f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",

    "fact_daily_sample":
        f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",

    "fact_query_90d":
        f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

print("Connected to the FlyRank warehouse.")

Connected to the FlyRank warehouse.


In [39]:
con.sql(
    f"SELECT COUNT(*) AS total_rows FROM {TABLES['fact_daily']}"
).df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows
0,78835655


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row represents one content item for one client on one reporting date.

I will use `fact_content_daily_performance` as the main table and `dim_content` when content metadata is needed.

I will develop and verify on the March 2026 partition (`month = '2026-03'`).

I will rank content items by refresh opportunity using observed search-performance signals as decision-support evidence.

I deliberately exclude future or label-derived information, especially `trend_pct` and `trend_direction`, because these fields contain outcome information and would cause leakage if used as features.

In [40]:
query_grain = f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    COUNT(*) AS row_count
FROM {TABLES['fact_daily']}
WHERE month = '2026-03'
GROUP BY
    report_date,
    client_hash_id,
    content_hash_id
HAVING COUNT(*) > 1
LIMIT 5
"""

grain_check = con.execute(query_grain).fetchdf()

grain_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,row_count


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Features

- `gsc_impressions` — historical Google Search Console impressions.
- `gsc_clicks` — historical Google Search Console clicks.
- `gsc_avg_position` — historical average search position.
- `content_age_days` — age of the content at the decision moment.
- `days_since_last_update` — time since the content was last updated.

### Label / proxy

- Future decline in search performance is the outcome concept. For the leakage experiment, I will deliberately use `trend_pct` as a label-derived field and then remove it.

### Context

- `client_hash_id` — identifies the client for grouping and joins.
- `content_hash_id` — identifies the content item for grouping and joins.
- `report_date` — identifies the observation date.
- `month` — identifies the warehouse partition.

### Excluded

- `trend_pct` — excluded because it is label-derived.
- `trend_direction` — excluded because it is used to define the decline outcome.
- Client/content IDs — used for grouping and joins, not predictive features.
- `ga4_data_available` — used for availability checking, not as a predictive feature.
- `_sample` — excluded because it contains the final June 2026 month.

In [41]:
schema_check = con.sql(
    f"""
    SELECT *
    FROM {TABLES['fact_daily']}
    LIMIT 1
    """
).df()

print("Warehouse columns:")
print(schema_check.columns.tolist())


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Warehouse columns:
['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events', 'month']


## 3. Verify it with queries (grain, counts, missing values, windows)

The March 2026 partition is used for all verification below. The queries verify the claimed grain, the size and date span of the slice, and the availability of GA4 data.

In [42]:
query_grain = f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    COUNT(*) AS row_count
FROM {TABLES['fact_daily']}
WHERE month = '2026-03'
GROUP BY
    report_date,
    client_hash_id,
    content_hash_id
HAVING COUNT(*) > 1
LIMIT 5
"""

grain_check = con.execute(query_grain).fetchdf()

print("Duplicate rows at the claimed grain:")
grain_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Duplicate rows at the claimed grain:


,report_date,client_hash_id,content_hash_id,row_count


In [43]:
query_window = f"""
SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS min_date,
    MAX(report_date) AS max_date
FROM {TABLES['fact_daily']}
WHERE month = '2026-03'
"""

window_check = con.execute(query_window).fetchdf()

print("March 2026 row count and date span:")
window_check

March 2026 row count and date span:


,row_count,min_date,max_date
0,9841378,2026-03-01,2026-03-31


In [44]:
query_availability = f"""
SELECT
    COUNT(*) AS available_rows
FROM {TABLES['fact_daily']}
WHERE month = '2026-03'
  AND ga4_data_available IS TRUE
"""

availability_check = con.execute(query_availability).fetchdf()

print("Rows where GA4 data is available:")
availability_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows where GA4 data is available:


,available_rows
0,413966


### Five-feature frame

I use five historical features from the March 2026 daily performance data:

1. `gsc_impressions` — available when the historical Google Search Console observation has already been recorded.
2. `gsc_clicks` — available when the historical Google Search Console observation has already been recorded.
3. `gsc_avg_position` — available when the historical Search Console position has already been observed.
4. `ga4_pageviews` — available when the historical GA4 measurement has been recorded and `ga4_data_available` is TRUE.
5. `ga4_sessions` — available when the historical GA4 measurement has been recorded and `ga4_data_available` is TRUE.

These are historical measurements and therefore can be known at a decision moment after the relevant reporting period has closed.

I exclude `trend_pct` and `trend_direction` because they are not columns in the daily warehouse table available for this lane and would represent outcome-derived information if used as features.

In [45]:
feature_query = f"""
SELECT
    client_hash_id,
    content_hash_id,
    report_date,
    gsc_impressions,
    gsc_clicks,
    gsc_avg_position,
    ga4_pageviews,
    ga4_sessions
FROM {TABLES['fact_daily']}
WHERE month = '2026-03'
  AND gsc_data_available IS TRUE
LIMIT 1000
"""

feature_df = con.execute(feature_query).fetchdf()

feature_df.head(10)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,report_date,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_pageviews,ga4_sessions
0,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,2026-03-01,20,0,3.350000,<NA>,<NA>
1,client_73cda7b4e4f265ea,content_05597932fe4da067,2026-03-01,1,0,0.000000,<NA>,<NA>
2,client_73cda7b4e4f265ea,content_7a105f548d9c6916,2026-03-01,125,1,4.928000,<NA>,<NA>
3,client_73cda7b4e4f265ea,content_905aa32a0230694e,2026-03-01,7,0,4.000000,<NA>,<NA>
4,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,2026-03-01,11,0,2.272727,<NA>,<NA>
5,client_73cda7b4e4f265ea,content_36c36abc7650d7af,2026-03-01,239,1,7.347280,<NA>,<NA>
6,client_73cda7b4e4f265ea,content_a7da352b73b02668,2026-03-01,191,0,7.832461,<NA>,<NA>
7,client_73cda7b4e4f265ea,content_05434271b257bb68,2026-03-01,55,0,3.272727,<NA>,<NA>
8,client_73cda7b4e4f265ea,content_d056587ff7faca0c,2026-03-01,77,0,5.636364,<NA>,<NA>
9,client_73cda7b4e4f265ea,content_bfd1e41c2af250c8,2026-03-01,2,0,4.500000,<NA>,<NA>


### Why these features are available when the decision is made

- `gsc_impressions`: known after the relevant Search Console reporting period has been observed.
- `gsc_clicks`: known after the relevant Search Console reporting period has been observed.
- `gsc_avg_position`: known after the relevant Search Console reporting period has been observed.
- `ga4_pageviews`: known after the relevant GA4 reporting period has been observed and the availability flag is TRUE.
- `ga4_sessions`: known after the relevant GA4 reporting period has been observed and the availability flag is TRUE.

The feature frame is restricted to March 2026 and does not use the final June 2026 `_sample` table.

In [46]:
label_query = f"""
WITH march AS (
    SELECT
        client_hash_id,
        content_hash_id,
        report_date,
        gsc_impressions
    FROM {TABLES['fact_daily']}
    WHERE month = '2026-03'
      AND gsc_data_available IS TRUE
),
with_future AS (
    SELECT
        *,
        LEAD(gsc_impressions) OVER (
            PARTITION BY client_hash_id, content_hash_id
            ORDER BY report_date
        ) AS next_day_impressions
    FROM march
)
SELECT
    client_hash_id,
    content_hash_id,
    report_date,
    gsc_impressions,
    next_day_impressions,
    CASE
        WHEN next_day_impressions < gsc_impressions THEN 1
        WHEN next_day_impressions >= gsc_impressions THEN 0
        ELSE NULL
    END AS is_declining_label
FROM with_future
WHERE next_day_impressions IS NOT NULL
LIMIT 10000
"""

label_df = con.execute(label_query).fetchdf()

label_df.head(10)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,report_date,gsc_impressions,next_day_impressions,is_declining_label
0,client_0797ff3a1fc9a6a5,content_c88d7630a340a086,2026-03-25,7,35,0
1,client_0797ff3a1fc9a6a5,content_c88d7630a340a086,2026-03-26,35,37,0
2,client_0797ff3a1fc9a6a5,content_c88d7630a340a086,2026-03-27,37,43,0
3,client_0797ff3a1fc9a6a5,content_c88d7630a340a086,2026-03-28,43,46,0
4,client_0797ff3a1fc9a6a5,content_c88d7630a340a086,2026-03-29,46,37,1
5,client_0797ff3a1fc9a6a5,content_c88d7630a340a086,2026-03-30,37,35,1
6,client_0797ff3a1fc9a6a5,content_c89645311e3a5d17,2026-03-02,4,7,0
7,client_0797ff3a1fc9a6a5,content_c89645311e3a5d17,2026-03-03,7,7,0
8,client_0797ff3a1fc9a6a5,content_c89645311e3a5d17,2026-03-05,7,3,1
9,client_0797ff3a1fc9a6a5,content_c89645311e3a5d17,2026-03-06,3,2,1


### Deliberate leakage experiment

For the leakage experiment, I deliberately use the label-derived outcome information as a feature. The purpose is not to build a valid model, but to demonstrate how using information derived from the outcome can make a quick score look artificially strong.

The leaked feature is removed afterward. It is not part of the final honest feature set.

In [47]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

leak_data = label_df.dropna(
    subset=["is_declining_label"]
).copy()

X_leak = leak_data[["is_declining_label"]]
y = leak_data["is_declining_label"]

X_train, X_test, y_train, y_test = train_test_split(
    X_leak,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

leak_model = DecisionTreeClassifier(
    max_depth=1,
    random_state=42
)

leak_model.fit(X_train, y_train)

leaked_score = accuracy_score(
    y_test,
    leak_model.predict(X_test)
)

print("Deliberately leaked score:", leaked_score)

Deliberately leaked score: 1.0


### Leakage removed

The deliberately leaked feature is the label itself, so its near-perfect score is not meaningful predictive performance. It demonstrates why label-derived information must never be included among model features.

I remove the leaked column and keep the final feature set limited to historical measurements that are available at the decision moment.

In [48]:
honest_feature_columns = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_pageviews",
    "ga4_sessions"
]

honest_features = feature_df[honest_feature_columns].copy()

print("Final honest features:")
print(honest_features.columns.tolist())

Final honest features:
['gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions']


## 4. Data limits

### Limitation

This warehouse slice has an unbalanced history, so different clients and content items may have different amounts of historical data.

Some rows have GSC data available while GA4 data is unavailable. Therefore, a missing GA4 measurement should not automatically be treated as zero activity; the availability flag needs to be checked.

A March 2026 slice provides observed historical performance, but it cannot prove that changing or refreshing content will cause future traffic or engagement to improve.

Historical windows can also overlap with outcome windows if they are not separated carefully. Therefore, features must be constructed only from information available before the prediction decision.

The resulting ranking should be treated as directional decision-support, not as a causal claim.

In [49]:
print("March 2026 limitation checks:")

print(
    "Rows with GSC available:",
    con.execute(
        f"""
        SELECT COUNT(*)
        FROM {TABLES['fact_daily']}
        WHERE month = '2026-03'
          AND gsc_data_available IS TRUE
        """
    ).fetchone()[0]
)

print(
    "Rows with GA4 available:",
    con.execute(
        f"""
        SELECT COUNT(*)
        FROM {TABLES['fact_daily']}
        WHERE month = '2026-03'
          AND ga4_data_available IS TRUE
        """
    ).fetchone()[0]
)

March 2026 limitation checks:
Rows with GSC available: 3611061
Rows with GA4 available: 413966


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.